In [0]:
import pandas as pd

# Use the /dbfs prefix for DBFS paths or direct volume paths
df_pandas = pd.read_csv("/Volumes/carnot/volume_raw/metadata/parcel_metadata.csv")
df_spark = spark.createDataFrame(df_pandas)
# Write to table in carnot.bronze.metadata
df_spark.write.mode("overwrite").saveAsTable("carnot.bronze.metadata")

In [0]:
from pyspark.sql.functions import col, lower, when, to_date, coalesce, try_to_date

df_src = spark.table("carnot.bronze.parcel_readings").dropDuplicates(["parcel_id", "date"])

# Try multiple date formats and convert to 'dd-MM-yyyy' as date datatype
date_formats = ["yyyy-MM-dd", "dd/MM/yyyy", "dd-MMM-yyyy"]
parsed_date = coalesce(*[try_to_date(col("date"), fmt) for fmt in date_formats])

df_clean = df_src.withColumn("date", parsed_date).withColumn(
        "sensor_status",
        when(lower(col("sensor_status")) == "ok", "ok")
        .when(lower(col("sensor_status")) == "error", "error")
        .otherwise("unkown")
    ).withColumn(
        "ndvi_value",
        when((col("ndvi_value") >= -1) & (col("ndvi_value") <= 1), col("ndvi_value")).otherwise(None)
    ).withColumn(
        "ndvi_flag",
        when(col("ndvi_value").isNotNull(), 1).otherwise(0)
    )




In [0]:
df_metadata = spark.table("carnot.bronze.metadata")
df_joined = df_clean.join(df_metadata, on="parcel_id", how="left")
# df_joined.write.mode("overwrite").saveAsTable("carnot.silver.cleaned_parcel_timeseries")
display(df_joined)


parcel_id,date,ndvi_value,temperature_c,rainfall_mm,sensor_status,ndvi_flag,mill_id,crop_type,sowing_date,area_hectares
PARCEL_014,2026-01-27,0.457,27.6,0.0,ok,1,MILL_NORTH,sugarcane,2026-01-05,9.39
PARCEL_004,2026-02-10,0.81,25.0,0.0,ok,1,MILL_NORTH,sugarcane,2026-01-02,3.18
PARCEL_001,2026-01-16,0.231,34.5,0.0,ok,1,MILL_NORTH,sugarcane,2026-02-10,9.03
PARCEL_001,2026-01-22,null,29.3,0.0,error,0,MILL_NORTH,sugarcane,2026-02-10,9.03
PARCEL_021,2026-04-26,0.429,15.9,0.0,ok,1,MILL_EAST,wheat,2026-02-21,3.58
PARCEL_012,2026-02-26,0.281,23.5,0.0,ok,1,MILL_WEST,sugarcane,2026-03-01,6.85
PARCEL_018,2026-02-07,0.298,33.8,0.0,ok,1,MILL_SOUTH,sugarcane,2026-01-11,5.82
PARCEL_010,2026-01-22,0.339,34.0,0.0,ok,1,MILL_EAST,sugarcane,2026-01-07,7.44
PARCEL_019,2026-03-25,0.58,15.5,0.0,ok,1,MILL_SOUTH,sugarcane,2026-01-18,10.19
PARCEL_009,2026-01-22,0.195,18.1,1.6,ok,1,MILL_EAST,sugarcane,2026-02-18,1.57


In [0]:
from pyspark.sql.functions import datediff, avg, countDistinct

# Filter out bad sensor_status
df_valid = df_joined.filter(col("sensor_status") == "ok")

# Before sowing: date in [sowing_date - 30, sowing_date)
df_before = df_valid.filter(
    (datediff(col("sowing_date"), col("date")) > 0) &
    (datediff(col("sowing_date"), col("date")) <= 30)
)

# After sowing: date in (sowing_date, sowing_date + 30]
df_after = df_valid.filter(
    (datediff(col("date"), col("sowing_date")) > 0) &
    (datediff(col("date"), col("sowing_date")) <= 30)
)

# Aggregate mean NDVI and parcel count per crop_type
df_result = df_before.groupBy("crop_type").agg(
    avg("ndvi_value").alias("mean_ndvi_before"),
    countDistinct("parcel_id").alias("n_parcels")
).join(
    df_after.groupBy("crop_type").agg(
        avg("ndvi_value").alias("mean_ndvi_after")
    ),
    on="crop_type",
    how="outer"
).select("crop_type", "mean_ndvi_before", "mean_ndvi_after", "n_parcels")


display(df_result)


crop_type,mean_ndvi_before,mean_ndvi_after,n_parcels
sugarcane,0.17780524344569293,0.33523370786516854,19
soybean,0.17053846153846153,0.3132758620689655,4
wheat,0.1793125,0.3050238095238095,2
